# Lesson 9A: Association Rules Theory — Support, Confidence, Lift, and Apriori

## Introduction

Every lesson so far has worked with continuous feature vectors and a distance metric. Association rule mining is a different animal entirely: the data is a set of **transactions**, each one a set of discrete **items**, with no notion of distance at all. The question isn't "how similar are these two points?" — it's "which items tend to appear together?"

This is the classic **market basket analysis** problem: "customers who buy bread and butter also tend to buy milk." No labels, no distances, no clusters — just counting how often item combinations co-occur, then deciding which co-occurrences are meaningful rather than coincidental.

The central challenge is combinatorial: with $n$ distinct items, there are $2^n - 1$ possible non-empty itemsets. Checking every one against every transaction is intractable for realistic catalog sizes. The **Apriori algorithm** (Agrawal & Srikant, 1994) solves this with one elegant pruning insight: **any subset of a frequent itemset must itself be frequent.** Turn that around, and it says any *superset* of an *infrequent* itemset must also be infrequent — so once an itemset is ruled out, everything built on top of it can be ruled out too, without ever counting it.

In this lesson:
1. Frame transactions, itemsets, and the frequent-itemset problem
2. Derive support, confidence, and lift as the three core metrics
3. Prove and apply the Apriori principle (downward closure)
4. Implement Apriori from scratch: candidate generation, pruning, frequent itemset mining
5. Generate association rules from frequent itemsets
6. Cross-check against mlxtend's `apriori` implementation

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Transactions and Itemsets](#transactions)
4. [Support, Confidence, and Lift](#metrics)
5. [The Apriori Principle](#apriori-principle)
6. [From-Scratch Apriori: Frequent Itemset Mining](#apriori-scratch)
7. [Rule Generation](#rule-generation)
8. [Cross-Check Against mlxtend](#mlxtend)
9. [Visualizing Rules](#visualization)
10. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from typing import List, Dict, FrozenSet, Tuple
from mlxtend.frequent_patterns import apriori as mlxtend_apriori
from mlxtend.preprocessing import TransactionEncoder

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="transactions"></a>
## Transactions and Itemsets

### The Data Model

A **transaction database** $T = \{t_1, t_2, \ldots, t_m\}$ is a collection of transactions, where each transaction $t_i$ is a **subset** of a universe of items $I = \{i_1, i_2, \ldots, i_n\}$ — a shopping basket, a browsing session, a set of co-prescribed medications.

An **itemset** is simply any subset of $I$. A **k-itemset** has exactly $k$ items. There is no order, no quantity, no distance — just presence or absence.

### Synthetic Grocery Transactions

We'll build a small synthetic transaction dataset with deliberately engineered co-occurrence patterns (bread+butter+milk, chips+soda, diapers+beer) plus random noise items, so we know in advance which rules a correct implementation *should* surface.

In [ ]:
rng = np.random.RandomState(42)

items_pool = ['bread', 'butter', 'milk', 'eggs', 'cheese', 'chips', 'soda', 'diapers', 'beer', 'coffee', 'sugar', 'flour']

def generate_transaction() -> List[str]:
    basket = set()
    # Engineered co-occurrence patterns, each triggered independently
    if rng.random() < 0.35:
        basket.update(['bread', 'butter'])
        if rng.random() < 0.6:
            basket.add('milk')
    if rng.random() < 0.25:
        basket.update(['chips', 'soda'])
    if rng.random() < 0.15:
        basket.update(['diapers', 'beer'])
    if rng.random() < 0.3:
        basket.update(['coffee', 'sugar'])
    # Background noise: a few random extra items
    n_noise = rng.randint(0, 3)
    basket.update(str(item) for item in rng.choice(items_pool, size=n_noise, replace=False))
    return sorted(basket) if basket else [str(rng.choice(items_pool))]

transactions = [generate_transaction() for _ in range(2000)]

print(f"Total transactions: {len(transactions)}")
print(f"Example transactions:")
for t in transactions[:5]:
    print(f"  {t}")

<a name="metrics"></a>
## Support, Confidence, and Lift

### Support

The **support** of an itemset $X$ is the fraction of transactions that contain it:

$$\text{support}(X) = \frac{|\{t \in T : X \subseteq t\}|}{|T|}$$

Support measures how *common* an itemset is. A **minimum support threshold** filters out itemsets too rare to matter — this is also what makes Apriori tractable, since it's the basis for the pruning step.

### Confidence

For a rule $X \Rightarrow Y$ (if a transaction contains $X$, it also tends to contain $Y$), **confidence** measures how often that implication actually holds:

$$\text{confidence}(X \Rightarrow Y) = \frac{\text{support}(X \cup Y)}{\text{support}(X)} = P(Y \mid X)$$

High confidence sounds good, but it can be misleading: if $Y$ is bought in 90% of *all* transactions anyway, a rule "confidently" predicting $Y$ tells you nothing about $X$'s influence.

### Lift

**Lift** corrects for that by comparing observed co-occurrence to what independence would predict:

$$\text{lift}(X \Rightarrow Y) = \frac{\text{support}(X \cup Y)}{\text{support}(X) \cdot \text{support}(Y)} = \frac{\text{confidence}(X \Rightarrow Y)}{\text{support}(Y)}$$

Interpretation:
- $\text{lift} = 1$: $X$ and $Y$ are statistically independent — no real association
- $\text{lift} > 1$: $X$ and $Y$ co-occur **more** than chance — a genuine positive association
- $\text{lift} < 1$: $X$ and $Y$ co-occur **less** than chance — buying $X$ makes $Y$ *less* likely

Lift is symmetric ($\text{lift}(X \Rightarrow Y) = \text{lift}(Y \Rightarrow X)$), unlike confidence, which is directional.

<a name="apriori-principle"></a>
## The Apriori Principle

### The Combinatorial Problem

With $n = 12$ items in our catalog, there are $2^{12} - 1 = 4095$ possible non-empty itemsets. Real catalogs have thousands of items — brute-force enumeration is completely infeasible.

### Downward Closure

**Apriori principle**: if an itemset $X$ is frequent (support $\geq$ minsup), then **every subset** of $X$ is also frequent.

Proof: any transaction containing $X$ also contains every subset of $X$ (subsets can only appear in *at least* as many transactions as the full set). So $\text{support}(X' ) \geq \text{support}(X)$ for any $X' \subset X$, and if $X$ clears the threshold, so does $X'$.

### The Contrapositive Is the Pruning Rule

Turn it around: **if any subset of $X$ is infrequent, $X$ cannot be frequent either.** This means we never need to count the support of an itemset whose subset already failed the threshold — we can prune it out of consideration *before* scanning the data. This is what keeps Apriori tractable: instead of generating all $2^n - 1$ itemsets, it only ever generates candidates built from itemsets already known to be frequent.

<a name="apriori-scratch"></a>
## From-Scratch Apriori: Frequent Itemset Mining

### Algorithm

1. **$L_1$**: count every single item's support; keep those $\geq$ minsup
2. **Candidate generation ($C_{k+1}$)**: join pairs of frequent $k$-itemsets that share $k-1$ items to form $(k+1)$-item candidates
3. **Pruning**: discard any candidate that has a $k$-subset not in $L_k$ (Apriori principle)
4. **Support counting**: scan transactions, count support for surviving candidates, keep those $\geq$ minsup as $L_{k+1}$
5. Repeat until no new frequent itemsets are found

In [ ]:
def compute_support(itemset: FrozenSet[str], transactions: List[FrozenSet[str]]) -> float:
    count = sum(1 for t in transactions if itemset.issubset(t))
    return count / len(transactions)


def generate_candidates(frequent_itemsets: List[FrozenSet[str]], k: int) -> List[FrozenSet[str]]:
    """Join frequent (k-1)-itemsets sharing k-2 items into k-itemset candidates."""
    candidates = set()
    items = sorted(frequent_itemsets, key=lambda s: sorted(s))
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            union = items[i] | items[j]
            if len(union) == k:
                candidates.add(union)
    return list(candidates)


def prune_candidates(candidates: List[FrozenSet[str]], prev_frequent: set, k: int) -> List[FrozenSet[str]]:
    """Apriori principle: discard any candidate with a (k-1)-subset that wasn't frequent."""
    survivors = []
    for candidate in candidates:
        if all(frozenset(subset) in prev_frequent for subset in combinations(candidate, k - 1)):
            survivors.append(candidate)
    return survivors


def apriori_scratch(transactions: List[List[str]], min_support: float) -> Dict[FrozenSet[str], float]:
    """Full Apriori frequent-itemset mining from scratch."""
    transaction_sets = [frozenset(t) for t in transactions]
    all_items = sorted(set(item for t in transactions for item in t))

    frequent_itemsets: Dict[FrozenSet[str], float] = {}

    # L1: single items
    current_level = {}
    for item in all_items:
        itemset = frozenset([item])
        support = compute_support(itemset, transaction_sets)
        if support >= min_support:
            current_level[itemset] = support
    frequent_itemsets.update(current_level)

    k = 2
    while current_level:
        candidates = generate_candidates(list(current_level.keys()), k)
        candidates = prune_candidates(candidates, set(current_level.keys()), k)

        next_level = {}
        for candidate in candidates:
            support = compute_support(candidate, transaction_sets)
            if support >= min_support:
                next_level[candidate] = support

        frequent_itemsets.update(next_level)
        current_level = next_level
        k += 1

    return frequent_itemsets


min_support = 0.05
frequent_itemsets_scratch = apriori_scratch(transactions, min_support=min_support)

print(f"From-scratch Apriori: {len(frequent_itemsets_scratch)} frequent itemsets at min_support={min_support}")
for itemset, support in sorted(frequent_itemsets_scratch.items(), key=lambda kv: -kv[1])[:10]:
    print(f"  {sorted(itemset)}: support={support:.3f}")

<a name="rule-generation"></a>
## Rule Generation

For every frequent itemset with 2+ items, generate all possible rules $X \Rightarrow Y$ by splitting it into a non-empty antecedent $X$ and consequent $Y = \text{itemset} \setminus X$, then compute confidence and lift for each split. Keep rules clearing a minimum confidence threshold.

In [ ]:
def generate_rules(frequent_itemsets: Dict[FrozenSet[str], float], min_confidence: float) -> List[dict]:
    rules = []
    for itemset, itemset_support in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for antecedent_items in combinations(items, r):
                antecedent = frozenset(antecedent_items)
                consequent = itemset - antecedent
                antecedent_support = frequent_itemsets.get(antecedent)
                consequent_support = frequent_itemsets.get(consequent)
                if antecedent_support is None or consequent_support is None:
                    continue

                confidence = itemset_support / antecedent_support
                if confidence < min_confidence:
                    continue

                lift = confidence / consequent_support
                rules.append({
                    'antecedent': sorted(antecedent),
                    'consequent': sorted(consequent),
                    'support': itemset_support,
                    'confidence': confidence,
                    'lift': lift,
                })
    return rules


min_confidence = 0.5
rules_scratch = generate_rules(frequent_itemsets_scratch, min_confidence=min_confidence)
rules_df = pd.DataFrame(rules_scratch).sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Generated {len(rules_df)} rules at min_confidence={min_confidence}")

# Pairwise rules (single item -> single item) isolate the four engineered co-occurrence patterns cleanly
pairwise_rules = rules_df[
    rules_df['antecedent'].apply(len).eq(1) & rules_df['consequent'].apply(len).eq(1)
].reset_index(drop=True)
pairwise_rules.head(10)

### A Caveat: Lift Inflates for Compound Itemsets

Sorting *all* rules by lift (rather than just the pairwise ones above) surfaces something worth
flagging: rules combining two *unrelated* engineered patterns — say `{chips, sugar} ⇒ {coffee, soda}`,
mixing items from the independent chips+soda and coffee+sugar triggers — can show **higher** lift
than the genuine single-item pairs like `bread ⇒ butter`, even though there is no real dependency
between the two patterns by construction.

The reason: lift divides joint support by the *product* of the antecedent's and consequent's own
supports. For a compound split, both of those marginal supports are themselves already conjunctions
(e.g. $P(\text{chips} \cap \text{sugar})$), so their product shrinks faster than the joint support does
— inflating the ratio even under true independence between the two source patterns. **Lift is not
invariant to itemset size or the way an itemset gets split into antecedent/consequent** — a well-known
limitation, not a bug in the metric's definition. This is exactly why practitioners cross-check high-lift
rules against support (rare + high-lift is a red flag, not automatically a strong signal) and consider a
rule's antecedent/consequent sizes, not lift in isolation.

In [ ]:
compound_vs_pairwise = pd.concat([
    pairwise_rules.assign(kind='pairwise (engineered pattern)'),
    rules_df[~rules_df.index.isin(pairwise_rules.index)].head(4).assign(kind='compound (cross-pattern)'),
])[['kind', 'antecedent', 'consequent', 'support', 'confidence', 'lift']]

print(compound_vs_pairwise.to_string(index=False))
print("\n💡 The compound cross-pattern rules show higher lift despite combining two independently-triggered patterns — support stays low throughout, which is the tell")

<a name="mlxtend"></a>
## Cross-Check Against mlxtend

`mlxtend.frequent_patterns.apriori` is a well-established reference implementation. We one-hot encode the same transactions and compare the frequent itemsets it finds — including their exact support values — against our from-scratch result. Rule generation itself is a deterministic downstream computation (confidence and lift are just ratios of already-validated supports), so an exact itemset match is the meaningful correctness check for the whole pipeline.

In [ ]:
encoder = TransactionEncoder()
one_hot = encoder.fit_transform(transactions)
df_encoded = pd.DataFrame(one_hot, columns=encoder.columns_)

frequent_mlxtend = mlxtend_apriori(df_encoded, min_support=min_support, use_colnames=True)
itemsets_mlxtend = {frozenset(row['itemsets']): row['support'] for _, row in frequent_mlxtend.iterrows()}

print(f"mlxtend Apriori: {len(itemsets_mlxtend)} frequent itemsets at min_support={min_support}")

# Direct comparison: same itemsets, same supports?
same_itemsets = set(frequent_itemsets_scratch.keys()) == set(itemsets_mlxtend.keys())
max_support_diff = max(
    abs(frequent_itemsets_scratch[i] - itemsets_mlxtend[i]) for i in frequent_itemsets_scratch
)

print(f"Identical itemset sets: {same_itemsets}")
print(f"Max support difference: {max_support_diff:.6f}")

<a name="visualization"></a>
## Visualizing Rules

Plot the pairwise rules — one point per engineered pattern — in (support, confidence) space, colored by lift, to see at a glance which rules are both common (high support) and reliable (high confidence and lift). Restricting to pairwise rules keeps the plot readable and avoids the compound-itemset lift inflation just discussed.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    pairwise_rules['support'], pairwise_rules['confidence'],
    c=pairwise_rules['lift'], cmap='RdYlGn', s=150, edgecolors='k', linewidth=0.5, vmin=1,
)
plt.colorbar(sc, ax=ax, label='Lift')

for _, row in pairwise_rules.iterrows():
    label = f"{'+'.join(row['antecedent'])} → {'+'.join(row['consequent'])}"
    ax.annotate(label, (row['support'], row['confidence']), fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Association Rules: Support vs Confidence, Colored by Lift', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 All four engineered pairs show up with lift well above 1 — confirming each is a genuine association, not chance")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Association rules operate on transactions and itemsets** — discrete, unordered, no distance metric involved.
2. **Support, confidence, and lift answer three different questions**: how common is this itemset, how reliable is this implication, and is the association genuine or just what independence would predict anyway.
3. **The Apriori principle (downward closure)** — every subset of a frequent itemset is frequent — is what makes itemset mining tractable: infrequent itemsets prune out entire branches of the search before they're ever counted.
4. **Rule generation is a second pass** over frequent itemsets, splitting each into antecedent/consequent and filtering by confidence.
5. **Lift > 1 is the signal that matters** — high confidence alone can just mean the consequent is popular everywhere.

### When to Use Association Rule Mining

- ✅ Transaction/basket data: retail co-purchases, web session clickstreams, co-prescribed medications
- ✅ Interpretable, rule-based output is valuable (unlike a black-box recommender)
- ❌ Continuous features with meaningful distances — clustering or PCA fit better
- ❌ Very large item catalogs without a scalable variant (FP-Growth avoids Apriori's repeated database scans)

### Next Steps

In Lesson 9B (practical), we'll:
- Apply mlxtend's Apriori and FP-Growth to a larger, more realistic market-basket dataset
- Explore minimum support/confidence trade-offs at scale
- Extract and rank actionable business rules
- Compare Apriori's repeated-scan cost against FP-Growth's tree-based approach

You now understand frequent itemset mining and rule generation — the discrete-data counterpart to the continuous-space methods from every earlier lesson 🎯